# MLFlow

[MLFlow documentation](https://docs.databricks.com/aws/en/mlflow/tracking)

In [0]:
# Upgrade to the latest MLflow version to use MLflow 3.0 features
%pip install mlflow>=3.0 --upgrade
dbutils.library.restartPython()

Note: you may need to restart the kernel using %restart_python or dbutils.library.restartPython() to use updated packages.


Loading data and cleaning it up. [Load_wine](https://scikit-learn.org/stable/modules/generated/sklearn.datasets.load_wine.html) comes clean and ready to be used

In [0]:
from sklearn.datasets import load_wine
from sklearn.model_selection import train_test_split

data = load_wine()

X = data['data']
y = data['target']

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)


## Model tracking using MLFlow. 

Here we train [RandomForestClassifier](https://scikit-learn.org/stable/modules/generated/sklearn.ensemble.RandomForestClassifier.html) and log
- Training parameters
- Run metrics
- Trained model inside the run

Model will be available within Experiment -> Run

In [0]:
import logging
import mlflow
import mlflow.sklearn
from mlflow.models import infer_signature
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (
    precision_score, recall_score, f1_score,
    roc_auc_score, log_loss
)

logging.getLogger("mlflow.tracking.context.registry").setLevel(logging.ERROR)

# Connect to remote MLflow server
mlflow.set_tracking_uri("databricks")
mlflow.set_registry_uri("databricks-uc")
mlflow.set_experiment("/Users/luisjosepr@gmail.com/demo")

model_info = None

# Model Tracking
with mlflow.start_run() as run:
    # pipeline.fit or model.fit()
    rf = RandomForestClassifier(max_depth=2, random_state=0)
    rf.fit(X_train, y_train)

    # Log model parameters
    mlflow.log_params(rf.get_params())

    y_pred = rf.predict(X_test)
    y_proba = rf.predict_proba(X_test)

    # Log model metrics
    mlflow.log_metrics({
        "accuracy": rf.score(X_test, y_test),
        "precision": precision_score(y_test, y_pred, average="weighted"),
        "recall": recall_score(y_test, y_pred, average="weighted"),
        "f1": f1_score(y_test, y_pred, average="weighted"),
        "log_loss": log_loss(y_test, y_proba)
    })

    # Required to log model and then use them for inference
    signature = infer_signature(X_train, rf.predict(X_train))
    model_info = mlflow.sklearn.log_model(
        rf,
        name="model",
        signature=signature,
        input_example=X_train[:5]
    )

    mlflow.set_tag("Training Info", "Basic RandomForestClassifier demo")

If you are using MLflow Tracing, you can migrate your traces to Unity Catalog for unlimited storage, fine-grained access controls, and queryability from notebooks, SQL, and dashboards. Learn more: https://docs.databricks.com/aws/en/mlflow3/genai/tracing/migrate-traces-to-uc
🔗 View Logged Model at: https://dbc-1d437b34-4be9.cloud.databricks.com/ml/experiments/34625841570123/models/m-fe93813741df46fb8ca492ed418a24e7?o=2688605696821110


## Model registry and model alias in Databricks Unity Catalog

We register the trained model in Unity Catalog so we can use it in other Notebooks or in model serving

In [0]:
# Databricks Unity Catalog
# Make sure the catalog and schema exist
catalog : str = "ceia"
schema : str = "oaa_ia"
model_name : str = "rf_demo"

registered_model = mlflow.register_model(
    model_uri=model_info.model_uri,      # f"runs:/{run.info.run_id}/model"
    name=f"{catalog}.{schema}.{model_name}" 
)
print(f"Registered {registered_model.name} version {registered_model.version}")

Registered model 'ceia.oaa_ia.rf_demo' already exists. Creating a new version of this model...


Uploading artifacts:   0%|          | 0/12 [00:00<?, ?it/s]

Registered ceia.oaa_ia.rf_demo version 3


🔗 Created version '3' of model 'ceia.oaa_ia.rf_demo': https://dbc-1d437b34-4be9.cloud.databricks.com/explore/data/models/ceia/oaa_ia/rf_demo/version/3?o=2688605696821110


We assign the newest trained model an alias so we can easily identify the latest mlflow experiment run. By common convention, we use `champion` as our latest model alias.

In [0]:
from mlflow import MlflowClient

client = MlflowClient()
client.set_registered_model_alias(
    name=f"{catalog}.{schema}.{model_name}" ,
    alias="champion",
    version=registered_model.version
)